# Image Captioning — Spatial Attention + Beam Search (Full Version)

## Setup & Imports

In [ ]:

import os, json, re, random, time
from pathlib import Path
from typing import List
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

SPECIAL={'pad':'<pad>','bos':'<bos>','eos':'<eos>','unk':'<unk>'}


## Vocabulary

In [ ]:

class Vocabulary:
    def __init__(self, min_freq=5):
        self.freqs={}; self.stoi={}; self.itos=[]; self.min_freq=min_freq

    def tokenize(self, text):
        text=text.lower()
        return re.findall(r"[a-zA-Z]+|\d+|[^\s\w]", text)

    def build(self, sentences):
        for s in sentences:
            for t in self.tokenize(s): self.freqs[t]=self.freqs.get(t,0)+1
        self.itos=[SPECIAL['pad'],SPECIAL['bos'],SPECIAL['eos'],SPECIAL['unk']]
        self.stoi={t:i for i,t in enumerate(self.itos)}
        for tok,f in sorted(self.freqs.items(), key=lambda x:(-x[1],x[0])):
            if f>=self.min_freq and tok not in self.stoi:
                self.stoi[tok]=len(self.itos); self.itos.append(tok)
        print('Vocab size:', len(self.itos))

    def numericalize(self, toks):
        return [self.stoi.get(t,self.stoi[SPECIAL['unk']]) for t in toks]

    def denumericalize(self, ids):
        return [self.itos[i] for i in ids]


## Dataset (simple Flickr8k only for demo)

In [ ]:

class Flickr8k(Dataset):
    def __init__(self, root, split, vocab, max_len=20):
        self.root=Path(root)
        self.vocab=vocab
        self.max_len=max_len
        cap_path=self.root/'captions'/'Flickr8k.token.txt'
        self.img_dir=self.root/'images'
        image2caps={}
        for line in open(cap_path):
            k,c=line.strip().split('	'); img=k.split('#')[0]
            image2caps.setdefault(img,[]).append(c)
        split_list=[x.strip() for x in open(self.root/'captions'/f'Flickr_8k.{split}Images.txt')]
        self.pairs=[(fn,cap) for fn in split_list for cap in image2caps.get(fn,[])]
        self.tf=transforms.Compose([
            transforms.Resize((256,256)), transforms.CenterCrop((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])

    def __len__(self): return len(self.pairs)

    def __getitem__(self,idx):
        fn,cap=self.pairs[idx]
        img=Image.open(self.img_dir/fn).convert('RGB')
        img=self.tf(img)
        toks=[SPECIAL['bos']] + self.vocab.tokenize(cap) + [SPECIAL['eos']]
        ids=self.vocab.numericalize(toks)
        ids=ids[:self.max_len]; length=len(ids)
        return img, torch.tensor(ids), length

def pad_collate(batch):
    imgs,seqs,lens=zip(*batch)
    imgs=torch.stack(imgs)
    maxlen=max(lens)
    pad_id=0
    out=torch.full((len(seqs),maxlen),pad_id,dtype=torch.long)
    for i,s in enumerate(seqs): out[i,:len(s)] = s
    return imgs, out, torch.tensor(lens)


## Spatial Attention + Decoder

In [ ]:

class SpatialEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        m=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        layers=list(m.children())[:-2]  # output: Bx2048x7x7
        self.cnn=nn.Sequential(*layers)
        self.adapt=nn.Conv2d(2048,512,1)

    def forward(self,x):
        fmap=self.cnn(x)          # B,C,H,W
        fmap=self.adapt(fmap)     # B,512,H,W
        B,C,H,W=fmap.shape
        feats=fmap.permute(0,2,3,1).view(B,H*W,C) # B,T,C
        return feats, (H,W)

class SpatialAttention(nn.Module):
    def __init__(self, feat_dim, hidden_dim):
        super().__init__()
        self.W=nn.Linear(feat_dim, hidden_dim)
        self.U=nn.Linear(hidden_dim, hidden_dim)
        self.v=nn.Linear(hidden_dim,1)

    def forward(self, feats, hidden):
        # feats: B,T,C ; hidden: B,H
        score=self.v(torch.tanh(self.W(feats)+self.U(hidden).unsqueeze(1)))
        alpha=torch.softmax(score, dim=1) # B,T,1
        ctx=(alpha*feats).sum(1)          # B,C
        return ctx, alpha

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, feat_dim=512):
        super().__init__()
        self.embed=nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attn=SpatialAttention(feat_dim, hidden_dim)
        self.lstm=nn.LSTMCell(embed_dim+feat_dim, hidden_dim)
        self.fc=nn.Linear(hidden_dim, vocab_size)

    def forward(self, feats, caps):
        B,Tv=feats.size(0), caps.size(1)
        hidden=torch.zeros(B,512,device=device)
        cell=torch.zeros(B,512,device=device)
        emb=self.embed(caps[:,0])
        outputs=[]
        for t in range(1,Tv):
            ctx,_=self.attn(feats, hidden)
            inp=torch.cat([emb,ctx],dim=1)
            hidden,cell=self.lstm(inp,(hidden,cell))
            out=self.fc(hidden)
            outputs.append(out.unsqueeze(1))
            emb=self.embed(caps[:,t])
        return torch.cat(outputs,dim=1)

    def beam_search(self, feats, bos, eos, beam=3, max_len=20):
        B,T,C=feats.shape
        hidden=torch.zeros(B,512,device=device)
        cell=torch.zeros(B,512,device=device)
        sequences=[([bos],0.0,hidden,cell)]
        for _ in range(max_len):
            all_cands=[]
            for seq,score,h,c in sequences:
                if seq[-1]==eos:
                    all_cands.append((seq,score,h,c)); continue
                emb=self.embed(torch.tensor([seq[-1]],device=device))
                ctx,_=self.attn(feats, h)
                h2,c2=self.lstm(torch.cat([emb,ctx],dim=1),(h,c))
                logits=self.fc(h2)
                probs=F.log_softmax(logits,dim=1)
                topk=torch.topk(probs,beam)
                for i in range(beam):
                    tok=topk.indices[0][i].item()
                    sc=score+topk.values[0][i].item()
                    all_cands.append((seq+[tok],sc,h2,c2))
            sequences=sorted(all_cands,key=lambda x:x[1],reverse=True)[:beam]
        return sequences[0][0]


## Save Notebook